In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
import joblib

# Project directories
ROOT = Path.cwd()

if ROOT.name == "notebooks":
    ROOT = ROOT.parent

DATA_PATH = ROOT / "data" / "raw" / "creditcard.csv"
PROCESSED_PATH = ROOT / "data" / "processed"
MODEL_PATH = ROOT / "models"

PROCESSED_PATH.mkdir(parents=True, exist_ok=True)
MODEL_PATH.mkdir(parents=True, exist_ok=True)

# Load dataset
df = pd.read_csv(DATA_PATH)

print("Dataset loaded:", df.shape)

Dataset loaded: (284807, 31)


In [3]:
duplicate_mask = df.duplicated()

print("Total duplicate rows:", duplicate_mask.sum())

print("\nDuplicate rows by class:")
print(df.loc[duplicate_mask, "Class"].value_counts())

Total duplicate rows: 1081

Duplicate rows by class:
Class
0    1062
1      19
Name: count, dtype: int64


In [4]:
df = df.drop_duplicates().reset_index(drop=True)

print("Shape after removing duplicates:", df.shape)

print("\nUpdated class distribution:")
print(df["Class"].value_counts())

Shape after removing duplicates: (283726, 31)

Updated class distribution:
Class
0    283253
1       473
Name: count, dtype: int64


In [5]:
X = df.drop("Class", axis=1)
y = df["Class"]

print("Features shape:", X.shape)
print("Target shape:", y.shape)

print("\nTarget distribution:")
print(y.value_counts())

Features shape: (283726, 30)
Target shape: (283726,)

Target distribution:
Class
0    283253
1       473
Name: count, dtype: int64


In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)

print("\nTraining class distribution:")
print(y_train.value_counts())

print("\nTesting class distribution:")
print(y_test.value_counts())

Training features: (226980, 30)
Testing features: (56746, 30)

Training class distribution:
Class
0    226602
1       378
Name: count, dtype: int64

Testing class distribution:
Class
0    56651
1       95
Name: count, dtype: int64


In [7]:
scaler = StandardScaler()

scale_columns = ["Time", "Amount"]

# Fit and transform training data
X_train = X_train.copy()
X_test = X_test.copy()

X_train[scale_columns] = scaler.fit_transform(
    X_train[scale_columns]
)

# Transform testing data using training scaler
X_test[scale_columns] = scaler.transform(
    X_test[scale_columns]
)

print("Feature scaling completed!")

print("\nTraining data:")
display(X_train[scale_columns].describe())

print("\nTesting data:")
display(X_test[scale_columns].describe())

Feature scaling completed!

Training data:


,Time,Amount
count,2.269800e+05,2.269800e+05
mean,-1.522636e-16,7.487965e-17
std,1.000002e+00,1.000002e+00
min,-1.998400e+00,-3.596386e-01
25%,-8.561822e-01,-3.364866e-01
50%,-2.123526e-01,-2.697972e-01
75%,9.363191e-01,-4.287453e-02
max,1.640237e+00,7.962087e+01



Testing data:


,Time,Amount
count,56746.000000,56746.000000
mean,-0.009386,0.001741
std,0.999221,1.091001
min,-1.998358,-0.359639
25%,-0.860031,-0.337300
50%,-0.219639,-0.271811
75%,0.929386,-0.048378
max,1.640111,104.175192


In [8]:
joblib.dump(
    scaler,
    MODEL_PATH / "standard_scaler.pkl"
)

print("Scaler saved successfully!")

Scaler saved successfully!


In [9]:
smote = SMOTE(random_state=42)

X_train_smote, y_train_smote = smote.fit_resample(
    X_train,
    y_train
)

print("Before SMOTE:")
print(y_train.value_counts())

print("\nAfter SMOTE:")
print(pd.Series(y_train_smote).value_counts())

print("\nOriginal training shape:", X_train.shape)
print("Resampled training shape:", X_train_smote.shape)

Before SMOTE:
Class
0    226602
1       378
Name: count, dtype: int64

After SMOTE:
Class
0    226602
1    226602
Name: count, dtype: int64

Original training shape: (226980, 30)
Resampled training shape: (453204, 30)


In [10]:
# Save original scaled train/test datasets
joblib.dump(
    {
        "X_train": X_train,
        "X_test": X_test,
        "y_train": y_train,
        "y_test": y_test
    },
    PROCESSED_PATH / "train_test_data.pkl"
)

# Save SMOTE-resampled training data separately
joblib.dump(
    {
        "X_train_smote": X_train_smote,
        "y_train_smote": y_train_smote
    },
    PROCESSED_PATH / "smote_training_data.pkl"
)

print("All processed datasets saved successfully!")

All processed datasets saved successfully!
